In [2]:
import pandas as pd
import numpy as np
from pandas import lreshape
from scipy.cluster.hierarchy import weighted
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score, log_loss, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression, ElasticNet
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import VotingClassifier, VotingRegressor, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from tqdm import tqdm
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector


## Using Bagging Classifier with Sonar Datset

In [3]:
sonar = pd.read_csv("Sonar.csv")
le = LabelEncoder()
y = le.fit_transform(sonar["Class"])
X = sonar.drop("Class", axis=1)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25,stratify=y)

In [5]:
decision_tree = DecisionTreeClassifier()
knn=KNeighborsClassifier()
nb=GaussianNB()
bagg =BaggingClassifier(estimator=nb, n_estimators=10,random_state=25)
bagg.fit(X_train,y_train)
y_pred = bagg.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.62      0.74      0.68        34
           1       0.61      0.48      0.54        29

    accuracy                           0.62        63
   macro avg       0.62      0.61      0.61        63
weighted avg       0.62      0.62      0.61        63



## Evaluate with Individual Classifier

In [17]:
bagg =BaggingClassifier(estimator=knn, n_estimators=10)
bagg.fit(X_train,y_train)
y_pred = bagg.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.72      0.91      0.81        34
           1       0.85      0.59      0.69        29

    accuracy                           0.76        63
   macro avg       0.79      0.75      0.75        63
weighted avg       0.78      0.76      0.75        63



In [7]:
bagg =BaggingClassifier(estimator=decision_tree, n_estimators=10, random_state=25)
bagg.fit(X_train,y_train)
y_pred = bagg.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.67      0.94      0.78        34
           1       0.87      0.45      0.59        29

    accuracy                           0.71        63
   macro avg       0.77      0.69      0.69        63
weighted avg       0.76      0.71      0.69        63



In [18]:
bag=BaggingClassifier(estimator=nb,n_estimators=10)
bag.fit(X_train,y_train)
y_pred=bag.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.62      0.76      0.68        34
           1       0.62      0.45      0.52        29

    accuracy                           0.62        63
   macro avg       0.62      0.61      0.60        63
weighted avg       0.62      0.62      0.61        63



In [21]:
lr = LogisticRegression()
bag=BaggingClassifier(estimator=lr,n_estimators=10)
bag.fit(X_train,y_train)
y_pred=bag.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.97      0.85        34
           1       0.95      0.62      0.75        29

    accuracy                           0.81        63
   macro avg       0.85      0.80      0.80        63
weighted avg       0.84      0.81      0.80        63



## Using List of Estimator

In [26]:
est_list = [lr,decision_tree,nb,knn]
n_est =[10, 15, 25, 50]
scores = []
for e in tqdm(est_list):
    for n in n_est:
        bagg = BaggingClassifier(n_estimators=n,estimator=e,random_state=25)
        bagg.fit(X_train,y_train)
        y_pred_prob = bagg.predict_proba(X_test)
        scores.append([e,n,log_loss(y_test,y_pred_prob)])
df_scores = pd.DataFrame(scores, columns=['Estimator','B-samples','score'])
df_scores.sort_values( 'score')


100%|██████████| 4/4 [00:01<00:00,  2.56it/s]


,Estimator,B-samples,score
12,KNeighborsClassifier(),10,0.422282
13,KNeighborsClassifier(),15,0.431490
14,KNeighborsClassifier(),25,0.432707
15,KNeighborsClassifier(),50,0.447037
6,DecisionTreeClassifier(),25,0.475750
4,DecisionTreeClassifier(),10,0.477146
7,DecisionTreeClassifier(),50,0.490503
5,DecisionTreeClassifier(),15,0.502120
1,LogisticRegression(),15,0.534413
0,LogisticRegression(),10,0.537041
